<a href="https://colab.research.google.com/github/Yago-Coqueiro/Qdrant-FastCamp/blob/main/Qdrant_Pratica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Triagem de Pacientes com Busca Semântica (Qdrant)

Este notebook implementa um sistema de **triagem inteligente de pacientes** usando busca semântica vetorial com o Qdrant.

### Como funciona?
O paciente (ou atendente) descreve os sintomas em linguagem natural. O sistema converte essa descrição em um vetor semântico e busca os quadros clínicos mais similares na base de dados, retornando:
- O diagnóstico provável
- O nível de urgência
- A especialidade médica indicada
- A conduta recomendada

### Diferenciais deste projeto
- Busca semântica: entende o **significado** dos sintomas, não apenas palavras-chave
- Filtros por metadados: filtra por urgência ou especialidade
- Payload rico: cada quadro clínico carrega informações estruturadas (CID-10, urgência, especialidade, conduta)

---
>  **Aviso:** Este projeto é estritamente educacional. Não substitui avaliação médica profissional.

### 1. Instalação das Dependências

In [23]:
# Instalando o cliente Qdrant e o fastembed para geração de embeddings
# - qdrant-client: SDK oficial para interagir com o banco de dados vetorial Qdrant
# - fastembed: biblioteca leve para gerar embeddings localmente sem precisar de GPU
!pip install qdrant-client fastembed -q

### 2. Conexão com o Qdrant Cloud

Utilizamos o Qdrant Cloud como banco de dados vetorial. Substitua a `url` e a `api_key` pelas suas credenciais.

> Você pode criar uma conta gratuita em [cloud.qdrant.io](https://cloud.qdrant.io)

In [22]:
from qdrant_client import QdrantClient

# Conectando ao cluster Qdrant Cloud
# url: endpoint do seu cluster (encontrado no painel do Qdrant Cloud)
# api_key: chave de autenticação gerada no painel
client = QdrantClient(
    url="Sua_url",
    api_key="Sua_API_key",
)

print(" Conexão estabelecida com sucesso!")

 Conexão estabelecida com sucesso!


### 3. Criação da Collection

Uma **collection** no Qdrant é equivalente a uma tabela em bancos relacionais. Aqui configuramos:
- `size=384`: dimensão dos vetores gerados pelo modelo `BAAI/bge-small-en-v1.5`
- `distance=COSINE`: métrica de similaridade, quanto mais próximo de 1, mais similares os vetores

In [24]:
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME = "triagem_pacientes"

# Criando a collection com configuração de vetores
# Cada ponto (quadro clínico) será representado por um vetor de 384 dimensões
# A similaridade entre vetores será medida pelo cosseno do ângulo entre eles
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

print(f" Collection '{COLLECTION_NAME}' criada com sucesso!")

 Collection 'triagem_pacientes' criada com sucesso!


### 4. Base de Quadros Clínicos

Aqui montamos os dados que vão popular nossa collection. Cada quadro clínico tem 6 campos:

- `nome` — nome do quadro clínico
- `sintomas` — descrição dos sintomas (é esse texto que vira embedding)
- `cid10` — código CID-10
- `urgencia` — de 1 (baixa) a 4 (crítica)
- `especialidade` — especialidade médica indicada
- `conduta` — o que fazer no atendimento

In [25]:
# Base de quadros clínicos com metadados ricos
# Cada tupla representa: (nome, sintomas, cid10, urgencia, especialidade, conduta)
# urgencia: 1=Baixa | 2=Média | 3=Alta | 4=Crítica

quadros_clinicos = [
    (
        "Infarto Agudo do Miocárdio",
        "Dor intensa no peito irradiando para o braço esquerdo e mandíbula, sudorese fria, falta de ar, náusea e sensação de morte iminente",
        "I21", 4, "Cardiologia",
        "Acionar SAMU imediatamente. Manter paciente em repouso. Administrar AAS se não contraindicado."
    ),
    (
        "Acidente Vascular Cerebral (AVC)",
        "Fraqueza ou dormência súbita no rosto braço ou perna especialmente em um lado do corpo, confusão, dificuldade para falar ou compreender, perda de visão, dor de cabeça intensa sem causa aparente",
        "I63", 4, "Neurologia",
        "Emergência absoluta. Acionar SAMU. Anotar horário do início dos sintomas. Não administrar nada via oral."
    ),
    (
        "Crise Asmática Grave",
        "Falta de ar intensa, chiado no peito, dificuldade para falar frases completas, uso de musculatura acessória para respirar, cianose nos lábios",
        "J46", 4, "Pneumologia",
        "Oxigenoterapia imediata. Broncodilatador inalatório. Monitoramento contínuo da saturação."
    ),
    (
        "Sepse",
        "Febre alta ou temperatura muito baixa, frequência cardíaca elevada, respiração acelerada, confusão mental, pressão arterial baixa, pele com aparência marmórea",
        "A41", 4, "Clínica Médica",
        "Protocolo de sepse. Coleta de hemoculturas. Antibioticoterapia empírica imediata. Reposição volêmica."
    ),
    (
        "Apendicite Aguda",
        "Dor abdominal que começa ao redor do umbigo e migra para o lado inferior direito do abdômen, náusea, vômito, febre baixa, piora da dor ao movimento",
        "K35", 3, "Cirurgia Geral",
        "Avaliação cirúrgica urgente. Exames laboratoriais e de imagem. Manter em jejum."
    ),
    (
        "Crise Hipertensiva",
        "Pressão arterial muito elevada acima de 180x120, dor de cabeça intensa na nuca, tontura, visão turva, zumbido no ouvido",
        "I10", 3, "Cardiologia",
        "Monitoramento contínuo da PA. Anti-hipertensivo conforme protocolo. Ambiente calmo e repouso."
    ),
    (
        "Fratura com Suspeita de Lesão Vascular",
        "Dor intensa no membro, deformidade visível, extremidade fria e pálida, ausência de pulso distal, parestesia",
        "T14", 3, "Ortopedia",
        "Imobilização imediata. Avaliação vascular urgente. Não reduzir sem avaliação médica."
    ),
    (
        "Pneumonia Bacteriana",
        "Febre alta com calafrios, tosse produtiva com escarro purulento ou com sangue, dor torácica ao respirar, falta de ar, cansaço intenso",
        "J18", 3, "Pneumologia",
        "Radiografia de tórax. Antibioticoterapia. Avaliar necessidade de internação conforme CURB-65."
    ),
    (
        "Crise Epiléptica",
        "Convulsões com movimentos involuntários do corpo, perda de consciência, mordedura de língua, incontinência urinária, confusão pós-ictal",
        "G40", 3, "Neurologia",
        "Proteger de lesões. Decúbito lateral. Benzodiazepínico se crise prolongada. Investigar causa."
    ),
    (
        "Cetoacidose Diabética",
        "Glicemia muito elevada, hálito com odor frutado, náusea e vômito, dor abdominal, respiração profunda e rápida, confusão mental, desidratação intensa",
        "E10", 3, "Endocrinologia",
        "Hidratação venosa vigorosa. Insulinoterapia. Correção de eletrólitos. Monitoramento intensivo."
    ),
    (
        "Infecção do Trato Urinário",
        "Ardência ao urinar, urgência miccional, aumento da frequência urinária, urina turva ou com sangue, dor suprapúbica leve",
        "N39", 1, "Urologia",
        "Urocultura. Antibioticoterapia oral. Aumento da ingestão hídrica. Retorno se febre ou piora."
    ),
    (
        "Gastroenterite Aguda",
        "Diarreia aquosa frequente, náusea, vômito, cólica abdominal, febre baixa, sinais de desidratação leve como boca seca e sede",
        "A09", 1, "Clínica Médica",
        "Hidratação oral. Dieta leve. Antieméticos e antidiarreicos conforme necessidade. Retorno se piora."
    ),
    (
        "Lombalgia Mecânica",
        "Dor nas costas na região lombar que piora com movimento e melhora com repouso, sem irradiação para membros, sem febre, relacionada a esforço físico",
        "M54", 1, "Ortopedia",
        "Analgesia e anti-inflamatório. Repouso relativo. Fisioterapia se necessário. Evitar esforços."
    ),
    (
        "Enxaqueca",
        "Cefaleia unilateral pulsátil de moderada a intensa intensidade, náusea, vômito, hipersensibilidade à luz e ao som, pode ter aura visual prévia",
        "G43", 2, "Neurologia",
        "Ambiente escuro e silencioso. Analgesia específica. Antieméticos. Investigar padrão de crises."
    ),
    (
        "Transtorno de Ansiedade com Crise de Pânico",
        "Palpitações, sudorese, tremores, sensação de falta de ar, tontura, medo intenso de morrer ou perder o controle, formigamento nas mãos, de início súbito",
        "F41", 2, "Psiquiatria",
        "Ambiente calmo. Técnicas de respiração. Ansiolítico se necessário. Encaminhar para acompanhamento."
    ),
    (
        "Conjuntivite Infecciosa",
        "Olho vermelho, secreção purulenta ou aquosa, sensação de areia nos olhos, lacrimejamento, leve fotofobia, sem perda de visão",
        "H10", 1, "Oftalmologia",
        "Higiene ocular. Colírio antibiótico se bacteriana. Orientar sobre contágio e isolamento."
    ),
    (
        "Hipoglicemia",
        "Tremores, suor frio, palidez, palpitações, sensação de fome intensa, confusão mental leve, fraqueza, tontura, pode evoluir para perda de consciência",
        "E16", 3, "Endocrinologia",
        "Glicose oral se consciente. Glucagon ou glicose IV se inconsciente. Identificar causa da hipoglicemia."
    ),
    (
        "Dengue",
        "Febre alta de início súbito, dor de cabeça intensa, dor atrás dos olhos, dores musculares e nas articulações, manchas vermelhas na pele, náusea",
        "A90", 2, "Infectologia",
        "Hidratação oral abundante. Paracetamol para febre. Evitar AAS e anti-inflamatórios. Hemograma seriado."
    ),
    (
        "Depressão Maior com Ideação Suicida",
        "Tristeza profunda persistente, perda de interesse em atividades, insônia ou hipersonia, pensamentos de morte ou suicídio, isolamento social, falta de energia",
        "F32", 3, "Psiquiatria",
        "Avaliação psiquiátrica urgente. Não deixar o paciente sozinho. Acionar rede de saúde mental. CVV: 188."
    ),
    (
        "Reação Alérgica Grave (Anafilaxia)",
        "Urticária generalizada, edema de face e garganta, dificuldade para engolir, falta de ar grave, queda da pressão arterial, após contato com alimento medicamento ou picada de inseto",
        "T78", 4, "Imunologia",
        "Adrenalina intramuscular imediata. Decúbito dorsal com pernas elevadas. Oxigênio. Acionar emergência."
    ),
]

print(f" Base de conhecimento carregada com {len(quadros_clinicos)} quadros clínicos")
print("\nDistribuição por urgência:")
for nivel, label in [(1, 'Baixa'), (2, 'Média'), (3, 'Alta'), (4, 'Crítica')]:
    count = sum(1 for q in quadros_clinicos if q[3] == nivel)
    print(f"  Urgência {nivel} ({label}): {count} quadros")

 Base de conhecimento carregada com 20 quadros clínicos

Distribuição por urgência:
  Urgência 1 (Baixa): 4 quadros
  Urgência 2 (Média): 3 quadros
  Urgência 3 (Alta): 8 quadros
  Urgência 4 (Crítica): 5 quadros


### 5. Geração de Embeddings e Indexação

Aqui convertemos os sintomas de cada quadro clínico em vetores numéricos (embeddings) e os inserimos no Qdrant junto com seus metadados.

**Por que indexar os sintomas e não o nome da doença?**
Porque o paciente não sabe o diagnóstico, ele descreve o que sente. Indexando a descrição dos sintomas, o sistema consegue encontrar matches semânticos com a queixa do paciente.

In [26]:
import warnings
warnings.filterwarnings("ignore")

from qdrant_client.models import PointStruct
from fastembed import TextEmbedding

# Carregando o modelo de embeddings
# BAAI/bge-small-en-v1.5: modelo leve e eficiente, gera vetores de 384 dimensões
print("Carregando modelo de embeddings...")
model = TextEmbedding('BAAI/bge-small-en-v1.5')

# Gerando embeddings a partir da descrição dos sintomas de cada quadro clínico
# É a descrição de sintomas que será comparada com a queixa do paciente
textos_para_embedding = [q[1] for q in quadros_clinicos]
embeddings = list(model.embed(textos_para_embedding))

# Construindo os pontos para inserção no Qdrant
# Cada ponto contém: id único, vetor semântico e payload com todos os metadados
pontos = []
for i, (embedding, quadro) in enumerate(zip(embeddings, quadros_clinicos)):
    nome, sintomas, cid10, urgencia, especialidade, conduta = quadro
    ponto = PointStruct(
        id=i,
        vector=embedding.tolist(),
        payload={
            "nome": nome,
            "sintomas": sintomas,
            "cid10": cid10,
            "urgencia": urgencia,           # int: 1=Baixa, 2=Média, 3=Alta, 4=Crítica
            "especialidade": especialidade, # str: usado para filtros
            "conduta": conduta,
        }
    )
    pontos.append(ponto)

# Inserindo todos os pontos na collection de uma vez (upsert)
# upsert = insert + update: insere se não existe, atualiza se já existe
resultado = client.upsert(
    collection_name=COLLECTION_NAME,
    points=pontos,
)

print(f" {len(pontos)} quadros clínicos indexados com sucesso!")
print(f"Status: {resultado.status}")

Carregando modelo de embeddings...
 20 quadros clínicos indexados com sucesso!
Status: completed


### 6. Função de Triagem

Criamos uma função reutilizável que encapsula toda a lógica de triagem. Ela aceita:
- A queixa do paciente em texto livre
- Filtros opcionais por urgência mínima e especialidade
- O número de resultados a retornar

In [27]:
from qdrant_client.models import Filter, FieldCondition, Range, MatchValue

def triar_paciente(queixa: str, urgencia_minima: int = None, especialidade: str = None, top_k: int = 3):
    """
    Realiza a triagem semântica de um paciente com base em sua queixa.

    Args:
        queixa: Descrição dos sintomas em linguagem natural
        urgencia_minima: Filtrar apenas quadros com urgência >= valor (1 a 4)
        especialidade: Filtrar apenas quadros de uma especialidade específica
        top_k: Número de resultados a retornar

    Returns:
        Lista de quadros clínicos similares com scores de similaridade
    """

    # Convertendo a queixa do paciente em vetor semântico
    vetor_queixa = next(iter(model.embed(queixa)))

    # Construindo filtros de metadados (opcional)
    # Os filtros permitem restringir a busca sem perder a semântica vetorial
    condicoes = []

    if urgencia_minima is not None:
        # Filtro de range: urgência maior ou igual ao valor mínimo informado
        condicoes.append(
            FieldCondition(key="urgencia", range=Range(gte=urgencia_minima))
        )

    if especialidade is not None:
        # Filtro de match exato: apenas quadros da especialidade informada
        condicoes.append(
            FieldCondition(key="especialidade", match=MatchValue(value=especialidade))
        )

    filtro = Filter(must=condicoes) if condicoes else None

    # Executando a busca vetorial com filtros
    resultados = client.query_points(
        collection_name=COLLECTION_NAME,
        query=vetor_queixa,
        query_filter=filtro,
        with_payload=True,
        limit=top_k,
    )

    return resultados.points


def exibir_triagem(queixa: str, resultados):
    """Formata e exibe os resultados da triagem de forma legível."""

    URGENCIA_LABELS = {1: " Baixa", 2: " Média", 3: " Alta", 4: " Crítica"}

    print("=" * 65)
    print(f" RESULTADO DA TRIAGEM")
    print(f" Queixa: {queixa}")
    print("=" * 65)

    for i, resultado in enumerate(resultados, 1):
        p = resultado.payload
        urgencia_label = URGENCIA_LABELS.get(p['urgencia'], str(p['urgencia']))
        similaridade = resultado.score * 100

        print(f"\n#{i} — {p['nome']} (CID-10: {p['cid10']})")
        print(f"   Similaridade  : {similaridade:.1f}%")
        print(f"   Urgência       : {urgencia_label}")
        print(f"   Especialidade  : {p['especialidade']}")
        print(f"   Conduta        : {p['conduta']}")
        print("-" * 65)

print(" Funções de triagem definidas e prontas para uso!")

 Funções de triagem definidas e prontas para uso!


### 7. Exemplos de Triagem

### 7.1 — Busca semântica simples
O paciente descreve os sintomas livremente. O sistema encontra os quadros mais similares.

In [28]:
# Exemplo 1: Queixa de dor no peito
# Note que o paciente não usa termos médicos — o sistema entende o significado
queixa_1 = "Estou com uma dor forte no peito que vai até o braço esquerdo, suando frio e com náusea"

resultados_1 = triar_paciente(queixa_1, top_k=3)
exibir_triagem(queixa_1, resultados_1)

 RESULTADO DA TRIAGEM
 Queixa: Estou com uma dor forte no peito que vai até o braço esquerdo, suando frio e com náusea

#1 — Infarto Agudo do Miocárdio (CID-10: I21)
   Similaridade  : 82.6%
   Urgência       :  Crítica
   Especialidade  : Cardiologia
   Conduta        : Acionar SAMU imediatamente. Manter paciente em repouso. Administrar AAS se não contraindicado.
-----------------------------------------------------------------

#2 — Apendicite Aguda (CID-10: K35)
   Similaridade  : 78.6%
   Urgência       :  Alta
   Especialidade  : Cirurgia Geral
   Conduta        : Avaliação cirúrgica urgente. Exames laboratoriais e de imagem. Manter em jejum.
-----------------------------------------------------------------

#3 — Cetoacidose Diabética (CID-10: E10)
   Similaridade  : 77.7%
   Urgência       :  Alta
   Especialidade  : Endocrinologia
   Conduta        : Hidratação venosa vigorosa. Insulinoterapia. Correção de eletrólitos. Monitoramento intensivo.
-----------------------------------

### 7.2 — Busca com filtro de urgência mínima
Útil para triagens em pronto-socorro, onde o sistema deve priorizar apenas quadros de alta ou crítica urgência.

In [29]:
from qdrant_client.models import PayloadSchemaType

# Criando índice no campo 'urgencia' — necessário para filtros do tipo Range (>=, <=)
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="urgencia",
    field_schema=PayloadSchemaType.INTEGER,
)

# Exemplo 2: Filtrando apenas quadros de urgência ALTA ou CRÍTICA (>= 3)
# Útil para triagem de pronto-socorro onde casos leves são descartados
queixa_2 = "Paciente apresenta confusão mental, febre muito alta, respiração acelerada e pressão baixa"

resultados_2 = triar_paciente(queixa_2, urgencia_minima=3, top_k=3)
exibir_triagem(queixa_2, resultados_2)

 RESULTADO DA TRIAGEM
 Queixa: Paciente apresenta confusão mental, febre muito alta, respiração acelerada e pressão baixa

#1 — Sepse (CID-10: A41)
   Similaridade  : 88.2%
   Urgência       :  Crítica
   Especialidade  : Clínica Médica
   Conduta        : Protocolo de sepse. Coleta de hemoculturas. Antibioticoterapia empírica imediata. Reposição volêmica.
-----------------------------------------------------------------

#2 — Hipoglicemia (CID-10: E16)
   Similaridade  : 77.9%
   Urgência       :  Alta
   Especialidade  : Endocrinologia
   Conduta        : Glicose oral se consciente. Glucagon ou glicose IV se inconsciente. Identificar causa da hipoglicemia.
-----------------------------------------------------------------

#3 — Cetoacidose Diabética (CID-10: E10)
   Similaridade  : 77.4%
   Urgência       :  Alta
   Especialidade  : Endocrinologia
   Conduta        : Hidratação venosa vigorosa. Insulinoterapia. Correção de eletrólitos. Monitoramento intensivo.
------------------------

### 7.3 — Busca com filtro por especialidade
Direciona a triagem para uma especialidade específica — por exemplo, em um ambulatório de neurologia.

In [30]:
from qdrant_client.models import PayloadSchemaType

# Criando índice no campo 'especialidade' — necessário para filtros do tipo MatchValue em strings
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="especialidade",
    field_schema=PayloadSchemaType.KEYWORD,
)

# Exemplo 3: Filtrando apenas quadros da especialidade de Neurologia
# Simula uma triagem dentro de um ambulatório especializado
queixa_3 = "Acordei sem conseguir mover o lado esquerdo do rosto e com dificuldade para falar"

resultados_3 = triar_paciente(queixa_3, especialidade="Neurologia", top_k=2)
exibir_triagem(queixa_3, resultados_3)

 RESULTADO DA TRIAGEM
 Queixa: Acordei sem conseguir mover o lado esquerdo do rosto e com dificuldade para falar

#1 — Acidente Vascular Cerebral (AVC) (CID-10: I63)
   Similaridade  : 75.9%
   Urgência       :  Crítica
   Especialidade  : Neurologia
   Conduta        : Emergência absoluta. Acionar SAMU. Anotar horário do início dos sintomas. Não administrar nada via oral.
-----------------------------------------------------------------

#2 — Crise Epiléptica (CID-10: G40)
   Similaridade  : 60.5%
   Urgência       :  Alta
   Especialidade  : Neurologia
   Conduta        : Proteger de lesões. Decúbito lateral. Benzodiazepínico se crise prolongada. Investigar causa.
-----------------------------------------------------------------


### 7.4 — Busca com múltiplos filtros combinados
Combinando urgência mínima e especialidade ao mesmo tempo.

In [31]:
# Exemplo 4: Combinando filtro de urgência E especialidade
# Útil para triagem avançada: "encontre quadros psiquiátricos de alta urgência"
queixa_4 = "Paciente relata pensamentos de se machucar, extremamente triste e isolado há semanas"

resultados_4 = triar_paciente(queixa_4, urgencia_minima=2, especialidade="Psiquiatria", top_k=2)
exibir_triagem(queixa_4, resultados_4)

 RESULTADO DA TRIAGEM
 Queixa: Paciente relata pensamentos de se machucar, extremamente triste e isolado há semanas

#1 — Depressão Maior com Ideação Suicida (CID-10: F32)
   Similaridade  : 77.7%
   Urgência       :  Alta
   Especialidade  : Psiquiatria
   Conduta        : Avaliação psiquiátrica urgente. Não deixar o paciente sozinho. Acionar rede de saúde mental. CVV: 188.
-----------------------------------------------------------------

#2 — Transtorno de Ansiedade com Crise de Pânico (CID-10: F41)
   Similaridade  : 69.0%
   Urgência       :  Média
   Especialidade  : Psiquiatria
   Conduta        : Ambiente calmo. Técnicas de respiração. Ansiolítico se necessário. Encaminhar para acompanhamento.
-----------------------------------------------------------------


### 8. Triagem Interativa

Experimente com sua própria queixa!

In [32]:
# Edite a queixa abaixo e rode a célula para testar o sistema
minha_queixa = "Estou com dor ao urinar, vontade de ir ao banheiro com frequência e urina turva"

# Parâmetros opcionais — defina None para não filtrar
minha_urgencia_minima = None      # Ex: 2 para urgência média ou superior
minha_especialidade = None        # Ex: "Cardiologia", "Neurologia", "Psiquiatria"...

resultados_interativo = triar_paciente(
    minha_queixa,
    urgencia_minima=minha_urgencia_minima,
    especialidade=minha_especialidade,
    top_k=3,
)
exibir_triagem(minha_queixa, resultados_interativo)

 RESULTADO DA TRIAGEM
 Queixa: Estou com dor ao urinar, vontade de ir ao banheiro com frequência e urina turva

#1 — Infecção do Trato Urinário (CID-10: N39)
   Similaridade  : 84.5%
   Urgência       :  Baixa
   Especialidade  : Urologia
   Conduta        : Urocultura. Antibioticoterapia oral. Aumento da ingestão hídrica. Retorno se febre ou piora.
-----------------------------------------------------------------

#2 — Crise Epiléptica (CID-10: G40)
   Similaridade  : 75.0%
   Urgência       :  Alta
   Especialidade  : Neurologia
   Conduta        : Proteger de lesões. Decúbito lateral. Benzodiazepínico se crise prolongada. Investigar causa.
-----------------------------------------------------------------

#3 — Infarto Agudo do Miocárdio (CID-10: I21)
   Similaridade  : 74.3%
   Urgência       :  Crítica
   Especialidade  : Cardiologia
   Conduta        : Acionar SAMU imediatamente. Manter paciente em repouso. Administrar AAS se não contraindicado.
-----------------------------------